<a href="https://colab.research.google.com/github/areebaeman234-ux/ML-Internship/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. My Lane

Lane 1 - Ranking Signal Analysis

Why I chose this:
I'm new to ML but have a statistics background. Lane 1 focuses on exploring data, finding patterns, and understanding relationships things I already know how to do with stats. This is a good starting point before building complex models.

What I want to discover:
Which signals (like content type, position, age) are associated with better page performance?

## 2. The Question

What I want to find out:
Which things (signals) make a page perform better in search results?
How this helps:
If I know what makes pages successful, content teams can focus on those things.

What someone will DO with this info:
- Write more content like the successful ones
- Improve weak pages
- Stop wasting time on things that don't matter

What happens if I'm wrong:
Teams waste time on wrong things. Pages still don't perform well.

Why data/stats helps:
There are thousands of pages. I can't look at all of them by eye. Statistics helps me find real patterns.

##3. Quick look at the data (2-3 real numbers)
Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.

In [1]:
import os
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"


if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        os.makedirs("data/raw", exist_ok=True)
        uploaded = files.upload()
        for fname in uploaded:
            os.replace(fname, DATA_PATH)
    except ImportError:
        pass

df = pd.read_csv(DATA_PATH)
print("raw rows:", len(df))

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
raw rows: 30000


In [2]:

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates(subset="content_id")

n_rows = len(df)
n_clients = df["client_id"].nunique()

pos_ctr_corr = df["avg_position"].corr(df["ctr"])

tier_ctr = df.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)

visible = df[df["impressions_90d"] >= 500]
pct_declining_visible = (visible["trend_direction"] == "down").mean() * 100

print(f"Rows after filtering: {n_rows}")
print(f"Distinct clients: {n_clients}")
print(f"Correlation(avg_position, ctr): {pos_ctr_corr:.3f}")
print()
print("Mean CTR by position_tier:")
print(tier_ctr)
print()
print(f"Visible pages (impressions_90d>=500): {len(visible)}")
print(f"% of visible pages trending down: {pct_declining_visible:.1f}%")



Rows after filtering: 30000
Distinct clients: 32
Correlation(avg_position, ctr): -0.073

Mean CTR by position_tier:
position_tier
top_3       1.483611
page_1      0.652467
striking    0.323239
page_3_5    0.222484
deep        0.150212
Name: ctr, dtype: float64

Visible pages (impressions_90d>=500): 16726
% of visible pages trending down: 59.6%


**Reading the numbers:**
- The raw linear correlation between `avg_position` and `ctr` is almost flat (-0.073) — on its own this would suggest position barely matters for CTR, which would be a misleading conclusion.
- Grouping into `position_tier` instead shows a ~10x spread in mean CTR (top_3 = 1.48 vs deep = 0.15). The signal is real, it's just non-linear/threshold-shaped, not a straight line — that's a concrete reason to prefer a tiered or model-based view over a naive correlation.
- 59.6% of "visible" pages (≥500 impressions in the last 90 days) are already trending down, out of 16,726 such pages in this 30k-row starter slice — so there's a large enough candidate pool that a review-priority ranking is genuinely useful, not just a toy exercise on a handful of pages.

## 4. Careful words: what I can and can't claim
I CAN say:
- Pages in top positions have higher CTR (1.48 vs 0.15)
- Position and CTR have a weak negative correlation (-0.073)
- These are patterns in THIS dataset

I CANNOT say:
- Position causes higher CTR
- This proves how Google works
- This applies to all websites

Why:
- Correlation ≠ Causation (I know this from my stats background!)
- There could be other factors I haven't seen
- This is just observation, not experimentation


## 5. Self-check

- [x] Picked one of the four predefined lanes: **Ranking Signal Analysis**
- [x] Named the decision (which pages a reviewer checks first) and the action (prioritize/deprioritize for review)
- [x] Showed at least two real numbers from the starter data (correlation=-0.073, 1.48 vs 0.15 tiered CTR spread, % declining visible pages)
- [x] Explained why this is not just "train a model" — the point is explaining *which* signals carry structure and why a naive linear view misses it, not just producing a score
- [x] Used careful language — separated what's observed on this slice from what's proven or generalizable to the full warehouse